# WEEK 8 - DEEP LEARNING

# 8.0 - ENVIRONMENT SETUP

## 8.0.1 - BUILD A NEW TENSOR FLOW ENVIRONMENT
BUILDING A NEW ENVIRONMENT IS RECOMMENDED AS THERE ARE A LOT OF CONFLICTS BETWEEN NEWER PACKAGES AND TTHE PACKAGES USED BY THE MOST CURRENT `tensorflow` PACKAGES.  ADDITIONALLY, THE  `tensorflow` PACKAGES ARE NOT COMPATIBLE WITH ALL PYTHON VERSION RELEASES \
\
```
        uv venv pytorch-env
        source pytorch-env/bin/activate     # Linux/macOS venv activation command
        .\pytorch-env\Scripts\activate      # Windows venv activation command
```

## 8.0.2 - INSTALL PACKAGES
1. OPEN A TERMINAL INSIDE YOUR NEW PYTHON ENVIRONMENT 
2. RUN THIS COMMAND TO INSTALL THE REQUIRED PACKAGES.  BE ADVISED THA THERR ARE DOZENS OF DEPENDENCIES 
\
        `uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128`
\ 

# 8.2 - JUPYTER NOTEBOOK SETUP

## 8.2.1 - CREATE A NEW JUPYTER NOTEBOOK

In [ ]:
# IMPORTS
import subprocess

import pandas     as pd 
from   PIL                   import Image
import numpy      as np

import torch
import tensorflow as tf
from   tensorflow            import keras

from   tensorflow.keras.preprocessing.image import load_img   # FUNCTION USED FOR LOADING IMAGES

In [ ]:
# GET DATASET
#   Dataset:
# 
#     Full:  https://github.com/alexeygrigorev/clothing-dataset
#     Small: https://github.com/alexeygrigorev/clothing-dataset-small
# 
#   Links:
#     https://cs231n.github.io/
#
# !git clone git@github.com:alexeygrigorev/clothing-dataset-small.git

from   pathlib               import Path
import subprocess

repo_dir         = Path("clothing-dataset-small")

if repo_dir.exists():
    print(f"Directory already exists: {repo_dir.resolve()}")
else:
    try:
        subprocess.run(
            [
                "git",                                                           # COMMAND
                "clone",                                                         # SUBROUTINE
                "https://github.com/alexeygrigorev/clothing-dataset-small.git",  # PARAMETER
            ],                                                                   # PROCESS TO RUN
            check          = True,                                               # RAISE AN ERROR MESSAGE IF ABORTS IN ERROR
            capture_output = True,                                               # CAPTURE `.stdout` and `.stderr`
            text           = True,                                               # RETURN CAPTURED OUTPUT AS A STRING INSTSEAD OF A BINARY
        )
        print("Repository cloned successfully.")

    except subprocess.CalledProcessError as e:
        print("Git clone failed:")
        print(e.stderr)



In [ ]:
img = Image.open('clothing-dataset-small/train/pants/0098b991-e36e-4ef1-b5ee-4154b21e2a92.jpg')
img.resize((256, 256))

In [ ]:
X = np.array(img.resize((224, 224)))


In [ ]:
import torchvision.models as models
from   torchvision import transforms

In [ ]:
# DOWNLOAD WEIGHTS FOR PRE-TRAINED MODEL 
model = models.mobilenet_v2(weights='IMAGENET1K_V1')
model.eval();

In [ ]:
preprocess =  transforms.Compose(
                [
                  transforms.Resize(256),
                  transforms.CenterCrop(224),
                  transforms.ToTensor(),
                  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ]
              )

In [ ]:
x = preprocess(img)

batch_t = torch.unsqueeze(x, 0)

with torch.no_grad():
    output = model(batch_t)

_, indices = torch.sort(output, descending=True)

In [ ]:
# THIS LINE DOES NOT RUN IN JUPYTER NOTEBOOK 
#    !wget https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt -O imagenet_classes.txt

# REPLACEMTN CODE 
from urllib.request import urlretrieve

url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
urlretrieve(url, "imagenet_classes.txt")

print("Downloaded imagenet_classes.txt")


In [ ]:
with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

# Get top 5 predictions
top5_indices = indices[0, :5].tolist()
top5_classes = [categories[i] for i in top5_indices]

print("Top 5 predictions:")
for i, class_name in enumerate(top5_classes):
    print(f"{i+1}: {class_name}")

In [ ]:
import os
from   torch.utils.data import Dataset

class ClothingDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
input_size = 224

# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

# Simple transforms - just resize and normalize
train_transforms = transforms.Compose(
                        [
                            transforms.RandomRotation(10),           # Rotate up to 10 degrees
                            transforms.RandomResizedCrop(
                                224, 
                                scale = (
                                            0.9, 
                                            1.0
                                        )
                            ),                                       # Zoom
                            transforms.RandomHorizontalFlip(),       # Horizontal flip
                            transforms.ToTensor(),
                            transforms.Normalize(
                                mean = mean, 
                                std  = std
                            )
                            
                        ]
                    )

val_transforms = transforms.Compose(
                    [
                        transforms.Resize(
                            (
                                input_size, 
                                input_size
                            )
                        ),
                        transforms.ToTensor(),
                        transforms.Normalize(
                            mean = mean, 
                            std  = std
                        )
                    ]
                  )

In [ ]:
from torch.utils.data import DataLoader

train_dataset = ClothingDataset(
    data_dir='./clothing-dataset-small/train',
    transform=train_transforms
)

val_dataset = ClothingDataset(
    data_dir='./clothing-dataset-small/validation',
    transform=val_transforms
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn

class ClothingClassifierMobileNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ClothingClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights='IMAGENET1K_V1')

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.output_layer = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)
        x = self.output_layer(x)
        return x

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ClothingClassifierMobileNet(num_classes=10)
model.to(device);

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [ ]:
1200 / 32

In [ ]:
# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    # Training phase
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Iterate over the training data
    for inputs, labels in train_loader:
        # Move data to the specified device (GPU or CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients to prevent accumulation
        optimizer.zero_grad()
        # Forward pass
        outputs = model(inputs)
        # Calculate the loss
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate training loss
        running_loss += loss.item()
        # Get predictions
        _, predicted = torch.max(outputs.data, 1)
        # Update total and correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate average training loss and accuracy
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # Validation phase
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    # Disable gradient calculation for validation
    with torch.no_grad():
        # Iterate over the validation data
        for inputs, labels in val_loader:
            # Move data to the specified device (GPU or CPU)
            inputs, labels = inputs.to(device), labels.to(device)
            # Forward pass
            outputs = model(inputs)
            # Calculate the loss
            loss = criterion(outputs, labels)

            # Accumulate validation loss
            val_loss += loss.item()
            # Get predictions
            _, predicted = torch.max(outputs.data, 1)
            # Update total and correct predictions
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate average validation loss and accuracy
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # Print epoch results
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

In [ ]:
def make_model(learning_rate=0.01):
    model = ClothingClassifierMobileNet(num_classes=10)
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer

In [ ]:
for lr in [0.001, 0.1]: #0.01, 0.1]:
  print("learning rate =", lr)
  model, optimizer = make_model(lr)
  train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)

In [ ]:
def train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device):
    best_val_accuracy = 0.0  # Initialize variable to track the best validation accuracy


    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total

        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            checkpoint_path = f'clothing_v4_{epoch+1:02d}_{val_acc:.3f}.pth'
            torch.save(model.state_dict(), checkpoint_path)
            print(f'Checkpoint saved: {checkpoint_path}')

In [ ]:
import torch.nn as nn

class ClothingClassifierMobileNet(nn.Module):
    def __init__(self, size_inner=100, droprate=0.2, num_classes=10):
        super(ClothingClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights='IMAGENET1K_V1')

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))

        self.inner = nn.Linear(1280, size_inner)  # New inner layer
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(droprate)  # Add dropout
        self.output_layer = nn.Linear(size_inner, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)
        x = self.inner(x)
        x = self.relu(x)
        x = self.dropout(x)  # Apply dropout
        x = self.output_layer(x)
        return x


def make_model(
        learning_rate=0.01,
        size_inner=100,
        droprate=0.2,
):
    model = ClothingClassifierMobileNet(
        num_classes=10,
        size_inner=size_inner,
        droprate=droprate,
    )
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer

In [ ]:
num_epochs = 50

model, optimizer = make_model(
    learning_rate=0.001,
    size_inner=100,
    droprate=0.2,
)

train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)

In [ ]:
!pip install torchsummary
from torchsummary import summary

summary(model, input_size=(3, 224, 224))

The output from torchsummary provides a detailed look at the layers in your ClothingClassifierMobileNet model:

    Layer (type): This column lists the type of each layer in the network (e.g., Conv2d, BatchNorm2d, ReLU6, InvertedResidual, Linear, AdaptiveAvgPool2d, Dropout).
    Output Shape: This shows the shape of the tensor that is output by each layer. The first dimension [-1] represents the batch size, which is variable. For convolutional layers, the shape is typically [batch_size, channels, height, width]. For linear layers, it's [batch_size, features].
    Param #: This column indicates the number of parameters in each layer. These are the weights and biases that the model learns during training.
    Total params: This is the total number of parameters in the entire model.
    Trainable params: This shows the number of parameters that will be updated during the training process. Since we froze the base MobileNetV2 layers (param.requires_grad = False), these parameters are not trainable.
    Non-trainable params: These are the parameters that are fixed and will not be updated during training. These are the parameters from the pre-trained MobileNetV2 base model.
    Input size (MB): The estimated size of the input data in megabytes.
    Forward/backward pass size (MB): The estimated memory required for the forward and backward passes through the network.
    Params size (MB): The estimated size of the model parameters in megabytes.
    Estimated Total Size (MB): The estimated total memory required by the model.

From this summary, we can see that:

    The base MobileNetV2 model has a large number of parameters, but they are non-trainable as intended.
    The trainable parameters are primarily in the new inner and output_layer that we added for our clothing classification task.
    The output shape of the final Linear-161 layer is [-1, 10], which corresponds to the 10 classes in our clothing dataset.


In [ ]:
path = '/content/clothing_v4_04_0.812.pth'

In [ ]:
model = ClothingClassifierMobileNet(size_inner=100, droprate=0.2, num_classes=10)
model.load_state_dict(torch.load(path))
model.to(device)
model.eval();

In [ ]:
x = val_transforms(img)
batch_t = torch.unsqueeze(x, 0).to(device)

with torch.no_grad():
    output = model(batch_t)

In [ ]:
dict(zip(classes, output[0].to('cpu')))

In [ ]:
classes = ['dress',
 'hat',
 'longsleeve',
 'outwear',
 'pants',
 'shirt',
 'shoes',
 'shorts',
 'skirt',
 't-shirt']

In [ ]:
!pip install onnx

In [ ]:
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Export to ONNX
onnx_path = "clothing_classifier_mobilenet_v2.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    verbose=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

In [ ]:
# GET DATASET
# Dataset:
# 
#     Full:  https://github.com/alexeygrigorev/clothing-dataset
#         C:\Users\duffl\anaconda3\_CT161_GitHub_Repos\ml-zoomcamp-homework\08_deep_learning
# 
#     Small: https://github.com/alexeygrigorev/clothing-dataset-small
# 
# Links:
#     https://cs231n.github.io/
# 
# !git clone git@github.com:alexeygrigorev/clothing-dataset-small.git

###########################################################################################
#   3.1.1 - GET DATA
###########################################################################################
data_path  = "C:/Users/duffl/anaconda3/_CT161_GitHub_Repos/ml-zoomcamp-homework/08_deep_learning/images.csv"
df         = pd.read_csv(data_path)



In [ ]:

# LOAD AN IMAGE

path             = 'C:/Users/duffl/anaconda3/_CT161_GitHub_Repos/clothing-dataset/images'
name             = '5f0a3fa0-6a3d-4b68-b213-72766a643de7.jpg'
fullname         = f'{path}/{name}'
load_img(fullname)

In [ ]:
from PIL import Image
import numpy as np

# Load an image
img = Image.open('clothing-dataset-small/train/pants/0098b991-e36e-4ef1-b5ee-4154b21e2a92.jpg')

# Resize to target size
img = img.resize((224, 224))

# Convert to numpy array
x = np.array(img)
print(x.shape)  # (224, 224, 3)